# Exercise 2: Kinematics

This notebook is a guided tutorial that combines:
- Understanding a kinematics computation library (pinocchio)
- Forward kinematics
- Numerical inverse kinematics (Jacobian-based)

## Learning goals
1. Familize yourself with the pinocchio library and its functionality for kinematics and jacobian.
2. Understand FK and how to validate IK with FK.
3. Track a moving target using IK.


In [12]:
import time
import numpy as np
import pinocchio as pin
import example_robot_data as RobData
from pinocchio.visualize import MeshcatVisualizer
import utils as Utils

ENABLE_VIEWER = True

## 1) Setup Robot and Viewer
After running the following code block, a web based visualization window should pop up, showing a tiago robot in home configuration.


TIPS: 
- Here in this script we use `q` for joint configurations instead of $\theta$ in the lecture and exercise.
- It's recommended to put code and visualization side by side to see the full process of visualization.

In [13]:
robot = RobData.load("tiago")
model = robot.model
data = model.createData()
q_home = robot.q0.copy()

EE_FRAME_ID = Utils.get_ee_frame_id(model)
BASE_FRAME_ID = robot.model.getFrameId('universe')

viz = None
viewer_enabled = False
if ENABLE_VIEWER:
    viz = MeshcatVisualizer(model, robot.collision_model, robot.visual_model)
    try:
        viz.initViewer(open=True)
        viz.loadViewerModel("tiago")
        viewer_enabled = True
    except Exception as e:
        print(f"[warning] Meshcat unavailable, fallback to headless: {e}")

if viewer_enabled:
    viz.display(q_home)

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7011/static/


Disclaimer: meshcat serves only as visualization tools, instead of physical simulation.

With Pinocchio you can probe the kinematic tree of the robot model. 

Each joint has its unique joint ID, with it you can access the stored kinematics information

In [14]:
for i, n in enumerate(robot.model.names):
    print(i, n)

0 universe
1 torso_lift_joint
2 arm_1_joint
3 arm_2_joint
4 arm_3_joint
5 arm_4_joint
6 arm_5_joint
7 arm_6_joint
8 arm_7_joint
9 hand_index_abd_joint
10 hand_index_virtual_1_joint
11 hand_index_flex_1_joint
12 hand_index_virtual_2_joint
13 hand_index_flex_2_joint
14 hand_index_virtual_3_joint
15 hand_index_flex_3_joint
16 hand_index_joint
17 hand_little_abd_joint
18 hand_little_virtual_1_joint
19 hand_little_flex_1_joint
20 hand_little_virtual_2_joint
21 hand_little_flex_2_joint
22 hand_little_virtual_3_joint
23 hand_little_flex_3_joint
24 hand_middle_abd_joint
25 hand_middle_virtual_1_joint
26 hand_middle_flex_1_joint
27 hand_middle_virtual_2_joint
28 hand_middle_flex_2_joint
29 hand_middle_virtual_3_joint
30 hand_middle_flex_3_joint
31 hand_mrl_joint
32 hand_ring_abd_joint
33 hand_ring_virtual_1_joint
34 hand_ring_flex_1_joint
35 hand_ring_virtual_2_joint
36 hand_ring_flex_2_joint
37 hand_ring_virtual_3_joint
38 hand_ring_flex_3_joint
39 hand_thumb_abd_joint
40 hand_thumb_virtual_1_

## 2) Kinematics Computation with pinocchio

you can also compute the forward kinematics easily, by first updating kinematic model using joint configuration

In [15]:
# update the kinematic tree using joint configuratiion
pin.framesForwardKinematics(model, data, q_home) 
pin.updateFramePlacements(model, data) # ! always neccessary to update the frame !

After updating kinematics model, you can query any joint or frame you like using the respective ID, for example for the end-effector or base we have

In [16]:
T_OE = data.oMf[EE_FRAME_ID]
T_OB = data.oMf[BASE_FRAME_ID]
print("End-effector pose:", T_OE)
print("Base pose:", T_OB)

End-effector pose:   R =
 4.89681e-12            1  4.68986e-12
          -1  4.89653e-12 -4.89669e-12
-4.89636e-12 -4.68942e-12            1
  p =   0.10805 -0.801075    0.7065

Base pose:   R =
1 0 0
0 1 0
0 0 1
  p = 0 0 0



and for the Jacobian of the end-effector pose (maps joint velocities to end-effector twist):

In [17]:
J = pin.computeFrameJacobian(robot.model, robot.data, q_home, EE_FRAME_ID, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
print(f'End-effector jacobian shape {J.shape}')
print(J)

End-effector jacobian shape (6, 48)
[[ 0.00000000e+00  8.15075000e-01  3.37904792e-12 -1.92356574e-12
  -3.51160450e-12 -6.23914767e-14  6.65750000e-02 -6.23405966e-14
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  1.50000000e-02 -4.86434865e-12 -1.55605199e-23
  -2.00000000e-02  3.87289874e-24  4.88928342e-13  9.86346071e-25
   0.00000000e+00  0.00000000e+00  0.00

### TODO 1: Build a Reusable FK Helper
Write a function that returns the end-effector pose as a homogeneous matrix $\mathbf{T}\in\mathbb{R}^{4\times4}$.

Why this matters:
- You will use this helper repeatedly to compute IK error.
- It keeps the IK code cleaner and easier to debug.

Note: data.oMf returns a `SE3` object can be converted into various formats. For the homogeneous transform, we'll need the homogeneous property ```data.oMf[ee_frame_id].homogeneous```

Expected output:
- `get_SE3_pose(...)` returns the homogeneous transform as `4x4` matrix.


In [18]:
def get_SE3_pose(model: pin.Model, data: pin.Data, q: np.ndarray, ee_frame_id: int) -> np.ndarray:
    # ===== TODO 1 =====
    return np.eye(4)
    # ===== END TODO =====

## 3) Inverse Kinematics
Now design a **vanilla numerical IK solver (position-only)** using the pseudoinverse of the end-effector position Jacobian. Note that pinocchio denotes the baseframe with $\{O\}$, while in the pen-and-paper exercise we directly used the world-frame $\{W\}$ since we consider a fixed-base in the exercise. Hence ```T_OE = T_WE```

General procedure:

a) Extract translation part (since we care only about position) ${}_W\mathbf{t}_{WE}$ from $ \mathbf{T}_{OE}$

$
~{}_W\mathbf{t}_{WE} = \mathbf{T}_{WE}[:3,3] (= \mathbf{T}_{OE}[:3,3])
$ 

b) Calulate the (remaining) position error
$
\mathbf{e} = {}_W\mathbf{t}_{WE,\text{goal}} - {}_W\mathbf{t}_{WE}
$.
Break if small enough (< tol).

c) Obtain the pseudoinverse for the current configuration ($\bm{\theta}$ in lecture, ```q``` in code)
$
\mathbf{J}(\bm{\theta})^{\dagger} = \mathbf{J}(\bm{\theta})^T(\mathbf{J}(\bm{\theta})\mathbf{J}(\bm{\theta})^T)^{-1}
$

d) Update
$
\mathbf{q} \leftarrow \mathbf{q} + \mathbf{J}^{\dagger}(k_p\mathbf{e})dt
$




In [19]:
def solve_numerical_ik_pos(
    model: pin.Model,
    data: pin.Data,
    ee_frame_id: int,
    q_init: np.ndarray,
    target_pos: np.ndarray,
    max_iters: int = 120,
    kp: float = 2.0,
    tol: float = 1e-3,
    dt: float = 0.08,
):
    q = q_init.copy()

    for _ in range(max_iters):
        ee_pos = get_SE3_pose(model, data, q, ee_frame_id)[:3, 3]

        # ===== TODO 2a =====
        # Cartesian position error, if error norm < tolerance, exit the loop
        # err = ...

        #if np.linalg.norm(err) < tol:
        break
        
        # ===== END TODO 2a =====

        # ===== TODO 2b =====
        # Position Jacobian + damped pseudo-inverse update
        # J = ...
        # dq = ...
        # ===== END TODO 2b =====

        dq = np.clip(dq, -0.2, 0.2)
        q = pin.integrate(model, q, dq * dt)

    final_err = np.linalg.norm(target_pos - get_SE3_pose(model, data, q, ee_frame_id)[:3, 3])
    return q, final_err


Now test the numerical solver on a static target.

Learning check:
- If convergence is good, position error is usually around `1e-3` m.
- If error is large, try tuning `kp`, `dt`, or `max_iters`. Mind workspace limits.


In [20]:
target_pos = np.array([0.55, 0.15, 0.35], dtype=float)
target_pose = T_OE.homogeneous.copy()
target_pose[:3, 3] = target_pos

Utils.create_sphere(viz, "world/target", radius=0.04, color_hex=0x8fce00)
if viewer_enabled:
    Utils.set_sphere_xyz(viz, "world/target", target_pos)

q_static, final_err = solve_numerical_ik_pos(
    model=model,
    data=data,
    ee_frame_id=EE_FRAME_ID,
    q_init = q_home.copy(),
    target_pos=target_pos
)
assert q_static.size != 0, "Numerical IK failed for static target"


if viewer_enabled:
    viz.display(q_static)

## 4) Tracking a Moving Target
Next, run numerical IK repeatedly along a trajectory.

Note:
- Use the previous solution as the next initial guess (`q_num`) to warm-start the iterative IK. This usually improves convergence speed and smoothness.


We first generate a small 3D figure-eight trajectory, then solve IK point-by-point.


In [21]:
traj_small = Utils.draw_eight_3D_trajectory(size=0.18, center=[0.55, 0.0, 0.35])

In [25]:
q_num = q_home.copy()
q_results = []
track_errors = []

for p in traj_small:
    pass
    # ===== TODO 3a =====
    # solve joint configuration for new target point p
    # q_num, err = ...
    # ===== END TODO 3a =====

    # ===== TODO 3b =====
    # compute error, log computed joint and error
    # q_results.append(...)
    # ee_pose = ...
    # track_errors.append(...)
    # ===== END TODO 3b =====

for p, q in zip(traj_small, q_results):
    if viewer_enabled:
        Utils.set_sphere_xyz(viz, "world/target", p)
        viz.display(q)
        time.sleep(0.03)

print("Numerical tracking samples:", len(track_errors))
print("Mean tracking error (m):", np.mean(track_errors))

if len(track_errors)>0:
    print("Max tracking error (m):", np.max(track_errors))


Numerical tracking samples: 0
Mean tracking error (m): nan


## 5) Out-of-Reach Trajectory

For bigger trajectories that are slightly out of reach, can IK solver still handle it? Feel free to tune the parameters of the trajectory and have a try!


In [23]:
traj_big = Utils.draw_eight_3D_trajectory(size=1.0, center=[0.75, 0.0, 0.55])

if viewer_enabled:
    Utils.create_sphere(viz, "world/big_target", radius=0.045, color_hex=0x8fce00)

q_num = q_home.copy()
numerical_errors = []

for p in traj_big:
    q_num, err = solve_numerical_ik_pos(
        model=model,
        data=data,
        ee_frame_id=EE_FRAME_ID,
        q_init=q_num,
        target_pos=p,
        max_iters=80,
        dt=0.08,
        kp=2.0,
    )
    numerical_errors.append(err)

    if viewer_enabled:
        # remove some previous sphere
        Utils.remove_node(viz, "world/target")
        Utils.set_sphere_xyz(viz, "world/big_target", p)
        viz.display(q_num)
        time.sleep(0.03)

print(f"Numerical IK min error (m): {np.min(numerical_errors):.4f}")
print(f"Numerical IK mean error (m): {np.mean(numerical_errors):.4f}")


Numerical IK min error (m): 0.8482
Numerical IK mean error (m): 1.1990


From the visualization we know that the robot failed to track waypoints out of the reach, this problem will be tackled in Ex3, where you have the chance to command the moving base of the robot.